In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

# Clean column names (removes hidden spaces/newlines)
df.columns = [col.strip() for col in df.columns]

# 2. Define Features and Target
# Using the sensor data as features
features = ['Air temperature [K]', 'Process temperature [K]', 
            'Rotational speed [rpm]', 'Torque [Nm]']
X = df[features]
y = df['Tool wear [min]'] # Regression Target

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train and Evaluate Models
degrees = [1, 2, 3] # Degree 1 is standard Linear Regression
results = {}

for deg in degrees:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=deg)),
        ('regressor', LinearRegression())
    ])
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[deg] = {'model': model, 'y_pred': y_pred, 'RMSE': rmse, 'R2': r2}
    print(f"Degree {deg}: RMSE = {rmse:.2f}, R² = {r2:.4f}")

# 5. Identify Best Model (Highest R2)
best_deg = max(results, key=lambda k: results[k]['R2'])
best_model_data = results[best_deg]
y_pred_best = best_model_data['y_pred']

# 6. Visualization
plt.figure(figsize=(12, 5))

# Plot 1: Actual vs Predicted
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_best, alpha=0.3, color='teal')
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--r', linewidth=2)
plt.title(f"Actual vs Predicted (Degree {best_deg})")
plt.xlabel("Actual Tool Wear")
plt.ylabel("Predicted Tool Wear")

# Plot 2: Residuals
plt.subplot(1, 2, 2)
residuals = y_test - y_pred_best
sns.histplot(residuals, kde=True, color='purple')
plt.axvline(0, color='red', linestyle='--')
plt.title(f"Residuals Distribution (Degree {best_deg})")
plt.xlabel("Error (Actual - Predicted)")

plt.tight_layout()
plt.show()

IncompleteRead: IncompleteRead(73581 bytes read)